# Day3_04A_CrewAI_Capstone_AI_Faculty_Assistant

## CrewAI Capstone Project - Building an AI Faculty Assistant

**Duration:** 35–45 minutes  
**Audience:** Engineering Faculty  
**Environment:** VS Code Notebook  
**Session:** Day 3 Capstone  

---

## Learning Objectives

By the end of this capstone notebook, participants will be able to:

- Design a complete multi-agent application using CrewAI.
- Create specialist Agents with clear role, goal, and backstory.
- Chain tasks using context.
- Use a custom tool inside a CrewAI workflow.
- Configure an LLM explicitly.
- Build a complete AI Faculty Assistant.
- Map the same architecture to enterprise use cases.

---

## Capstone Promise

Today you will not just run a demo.

You will build a small but complete AI application.

> An AI Faculty Assistant that prepares a complete teaching pack for tomorrow's class.


# 1. Project Story

Imagine this situation.

You are a professor at an engineering college.

Tomorrow morning, you need to teach:

> Agentic AI

But you have only 20 minutes to prepare.

You need:

- A lesson plan
- Teaching notes
- A quiz
- A lab exercise
- References
- Reviewer suggestions

Instead of preparing everything manually, you ask an AI team to help.

This is where CrewAI becomes powerful.


# 2. What We Will Build

We will build a CrewAI application with six specialist Agents.

```text
Faculty Topic
      │
      ▼
Research Specialist
      │
      ▼
Lesson Planner
      │
      ▼
Quiz Designer
      │
      ▼
Lab Exercise Designer
      │
      ▼
Reference Expert
      │
      ▼
Academic Reviewer
      │
      ▼
Final Faculty Teaching Pack
```

Each Agent has a clear responsibility.

This is the shift from:

```text
One prompt
```

to:

```text
A team of AI specialists
```


# 3. Concepts Revised in This Capstone

This notebook combines almost everything from the CrewAI section.

| Concept | Used in Capstone? |
|---|---|
| Agent | Yes |
| Task | Yes |
| Crew | Yes |
| Role, Goal, Backstory | Yes |
| Context Chaining | Yes |
| Sequential Process | Yes |
| Custom Tool | Yes |
| LLM Configuration | Yes |
| Final Output Formatting | Yes |

---

## Instructor Tip

Pause here and ask participants:

> Which of these concepts did we learn in the previous notebooks?

This helps them realize that the capstone is not new theory. It is integration.


# 4. Setup

Install dependencies if needed:

```bash
pip install crewai crewai-tools python-dotenv
```

Your project structure should look like this:

```text
FDP-LLM/
  .env
  requirements.txt
  day3-openai-crewai/
    Day3_04A_CrewAI_Capstone_AI_Faculty_Assistant.ipynb
```


In [ ]:
from pathlib import Path
from dotenv import load_dotenv
import os

from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool

# 5. Load Environment Variables

We will use the same root `.env` loading pattern used throughout the FDP.


In [ ]:
current = Path.cwd().resolve()

for folder in [current] + list(current.parents):
    env = folder / ".env"
    if env.exists():
        load_dotenv(env)
        break

print("OpenAI API Key Loaded:", os.getenv("OPENAI_API_KEY") is not None)

# 6. Configure the LLM

CrewAI is not the LLM.

CrewAI coordinates Agents, Tasks, Tools, and Crews.

The LLM provides the intelligence.

For live FDP demos, use a reliable and cost-effective model.


In [ ]:
llm = LLM(
    model="gpt-4o-mini"
)

print("LLM configured.")

# 7. Define the Teaching Topic

This is the input to our AI Faculty Assistant.

You can change this later during the live demo.


In [ ]:
teaching_topic = "Agentic AI for Engineering Students"

print("Teaching Topic:", teaching_topic)

# 8. Custom Tool: Teaching Resource Tool

We will keep the tool simple and safe.

This tool returns recommended learning resources for common AI topics.

In a real college system, this could connect to:

- LMS
- Library portal
- Course catalogue
- Internal document repository
- Faculty knowledge base


In [ ]:
@tool("Teaching Resource Tool")
def teaching_resource_tool(topic: str) -> str:
    """
    Returns recommended teaching resources for AI-related topics.
    Use this tool when an Agent needs books, documentation, references, or learning resources.
    """

    topic_lower = topic.lower()

    if "agent" in topic_lower:
        return (
            "Recommended resources for Agentic AI:\n"
            "1. OpenAI Agents SDK documentation\n"
            "2. CrewAI documentation\n"
            "3. LangGraph documentation for stateful agent workflows\n"
            "4. Classroom activity: Compare chatbot vs agent using a support scenario\n"
            "5. Suggested reading: Tool calling, planning, memory, and guardrails"
        )

    if "rag" in topic_lower or "retrieval" in topic_lower:
        return (
            "Recommended resources for RAG:\n"
            "1. LangChain RAG tutorials\n"
            "2. Vector database documentation such as Chroma, FAISS, or Pinecone\n"
            "3. Classroom activity: Build a simple document Q&A assistant\n"
            "4. Suggested reading: embeddings, chunking, retrieval, and evaluation"
        )

    if "prompt" in topic_lower:
        return (
            "Recommended resources for Prompt Engineering:\n"
            "1. OpenAI prompt engineering guide\n"
            "2. Classroom activity: Compare weak prompt vs improved prompt\n"
            "3. Suggested reading: role prompting, examples, constraints, and evaluation"
        )

    return (
        "General AI teaching resources:\n"
        "1. OpenAI documentation\n"
        "2. Google Machine Learning Crash Course\n"
        "3. Papers with Code for examples\n"
        "4. Classroom activity: Compare AI use cases across industries"
    )

## What did we just learn?

We created a simple read-only custom tool.

This tool is safe because it does not modify any system.

In enterprise AI, always begin with read-only tools before moving to action tools.


# 9. Test the Tool Directly

Always test tools before giving them to Agents.


In [ ]:
print(teaching_resource_tool.run("Agentic AI"))

# 10. Create the Six Specialist Agents

We now create:

1. Research Specialist
2. Lesson Planner
3. Quiz Designer
4. Lab Exercise Designer
5. Reference Expert
6. Academic Reviewer


In [ ]:
research_agent = Agent(
    role="AI Education Research Specialist",
    goal="Prepare concise, accurate, and classroom-friendly research notes for engineering faculty",
    backstory="""
    You are an AI education researcher who understands engineering classrooms,
    faculty development programs, and beginner-friendly technical teaching.
    You focus on clarity, practical examples, and conceptual correctness.
    """,
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

lesson_planner_agent = Agent(
    role="Faculty Lesson Planner",
    goal="Convert research notes into a clear 30-minute lesson plan",
    backstory="""
    You are an experienced engineering faculty trainer. You design lesson plans
    with introduction, core concepts, examples, classroom questions, and summary.
    You make topics easy for faculty to teach.
    """,
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

quiz_agent = Agent(
    role="Assessment and Quiz Designer",
    goal="Create useful quiz questions that check conceptual understanding",
    backstory="""
    You are an academic assessment designer. You create MCQs and short-answer
    questions that help faculty check whether students understood the topic.
    """,
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

lab_agent = Agent(
    role="Lab Exercise Designer",
    goal="Create a simple hands-on notebook activity for engineering students",
    backstory="""
    You design practical, low-code classroom exercises for AI and Agentic AI topics.
    Your activities are suitable for VS Code notebooks and students with basic Python knowledge.
    """,
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

reference_agent = Agent(
    role="Teaching Reference Expert",
    goal="Recommend useful books, documentation, videos, and learning resources",
    backstory="""
    You help faculty find practical teaching resources. You use available tools
    where appropriate and keep references simple and useful for classroom delivery.
    """,
    tools=[teaching_resource_tool],
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

reviewer_agent = Agent(
    role="Senior Academic Reviewer",
    goal="Review and improve the complete teaching pack for clarity, flow, and classroom readiness",
    backstory="""
    You are a senior academic reviewer and FDP mentor. You check whether the
    teaching pack is complete, simple, well-structured, and useful for engineering faculty.
    You provide final improvements and teaching tips.
    """,
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

# Pause & Reflect

We now have six specialist Agents.

| Agent | Responsibility |
|---|---|
| Research Specialist | Creates research brief |
| Lesson Planner | Creates lesson plan |
| Quiz Designer | Creates assessment |
| Lab Designer | Creates hands-on exercise |
| Reference Expert | Suggests resources |
| Academic Reviewer | Improves final pack |

This is the core idea of CrewAI:

> Think in terms of AI teams, not single prompts.


# 11. Create the Tasks

The workflow is sequential because each task builds on the previous outputs.

```text
Research → Lesson → Quiz → Lab → References → Review
```


In [ ]:
research_task = Task(
    description=f"""
    Research the topic: {teaching_topic}

    Prepare a concise research brief for engineering faculty.

    Include:
    1. Simple definition
    2. Why the topic matters
    3. Chatbot vs Agent comparison
    4. One DSU classroom example
    5. One enterprise example
    6. Common misconceptions
    """,
    expected_output="A concise research brief with clearly labelled sections.",
    agent=research_agent,
)

lesson_task = Task(
    description="""
    Using the research brief, create a 30-minute lesson plan.

    The lesson plan should include:
    1. Opening question
    2. Learning objectives
    3. Concept explanation
    4. Architecture explanation
    5. DSU classroom example
    6. Mini discussion activity
    7. Summary
    """,
    expected_output="A structured 30-minute lesson plan for engineering faculty.",
    agent=lesson_planner_agent,
    context=[research_task],
)

quiz_task = Task(
    description="""
    Create a short assessment based on the research brief and lesson plan.

    Include:
    1. Five MCQs with answers
    2. Two short-answer questions with expected points
    3. One classroom discussion question
    """,
    expected_output="A quiz section with MCQs, short answers, and discussion question.",
    agent=quiz_agent,
    context=[research_task, lesson_task],
)

lab_task = Task(
    description="""
    Design one simple hands-on notebook activity for students.

    The lab should include:
    1. Objective
    2. Concept being demonstrated
    3. Step-by-step activity
    4. Expected output
    5. Extension exercise

    Keep it simple for students with basic Python knowledge.
    """,
    expected_output="A practical lab exercise suitable for a VS Code notebook.",
    agent=lab_agent,
    context=[research_task, lesson_task],
)

reference_task = Task(
    description=f"""
    Recommend teaching resources for the topic: {teaching_topic}

    Use the Teaching Resource Tool where useful.

    Include:
    1. Documentation
    2. Suggested readings
    3. Classroom activity idea
    4. What faculty should explore next
    """,
    expected_output="A practical reference section for faculty.",
    agent=reference_agent,
    context=[research_task, lesson_task],
)

review_task = Task(
    description="""
    Review the complete teaching pack.

    You have access to:
    - Research brief
    - Lesson plan
    - Quiz
    - Lab exercise
    - References

    Improve the final output and provide:
    1. Final teaching pack summary
    2. Faculty delivery tips
    3. Any missing points
    4. Suggested improvements
    5. Final readiness verdict
    """,
    expected_output="A final reviewed teaching pack with reviewer suggestions.",
    agent=reviewer_agent,
    context=[research_task, lesson_task, quiz_task, lab_task, reference_task],
)

# 12. Create the Crew

We combine all Agents and Tasks into one Crew.


In [ ]:
faculty_assistant_crew = Crew(
    agents=[
        research_agent,
        lesson_planner_agent,
        quiz_agent,
        lab_agent,
        reference_agent,
        reviewer_agent,
    ],
    tasks=[
        research_task,
        lesson_task,
        quiz_task,
        lab_task,
        reference_task,
        review_task,
    ],
    process=Process.sequential,
    verbose=True,
)

# 13. Run the Capstone Crew

This may take a few minutes because six Agents are working in sequence.

During live teaching, explain that this is similar to a real team preparing content step by step.


In [ ]:
capstone_result = faculty_assistant_crew.kickoff()

print(capstone_result)

# 14. Beautiful Teaching Pack Formatter

The raw CrewAI result is useful, but for teaching we want a more polished display.


In [ ]:
def display_teaching_pack(topic: str, final_result):
    """
    Formats the final CrewAI result as a readable teaching pack.
    """

    line = "=" * 80
    section = "-" * 80

    print(line)
    print("AI FACULTY TEACHING PACK")
    print(line)
    print(f"Topic: {topic}")
    print(section)
    print("Generated using CrewAI Capstone Workflow")
    print(line)
    print()
    print("FINAL REVIEWED OUTPUT")
    print(section)
    print(final_result)
    print()
    print(line)
    print("END OF TEACHING PACK")
    print(line)

In [ ]:
display_teaching_pack(teaching_topic, capstone_result)

# 15. What Did We Just Build?

We built a complete AI Faculty Assistant.

```text
Faculty Topic
      ↓
Research Specialist
      ↓
Lesson Planner
      ↓
Quiz Designer
      ↓
Lab Exercise Designer
      ↓
Reference Expert
      ↓
Academic Reviewer
      ↓
Teaching Pack
```

This is not just a chatbot.

This is a team of AI specialists working toward a useful academic output.


# 16. Why This Is Better Than One Prompt

You could ask one LLM:

> Prepare a lesson plan, quiz, lab, and references.

But a Crew gives better structure.

| Single Prompt | CrewAI Workflow |
|---|---|
| One response | Multi-stage output |
| Less role clarity | Specialist roles |
| Hard to debug | Each task is visible |
| Hard to improve | Replace or improve one Agent |
| Limited review | Reviewer Agent checks quality |

---

## Instructor Tip

Ask participants:

> If the quiz quality is poor, which Agent or Task would you improve?

This teaches modular design.


# 17. Enterprise Mapping

The same architecture works outside education.

## Banking

```text
Customer Request
      ↓
Customer History Agent
      ↓
Risk Assessment Agent
      ↓
Compliance Agent
      ↓
Fraud Review Agent
      ↓
Manager Review
      ↓
Final Recommendation
```

## DevOps

```text
Production Alert
      ↓
Log Analysis Agent
      ↓
Metrics Agent
      ↓
Deployment Agent
      ↓
Security Agent
      ↓
RCA Writer
      ↓
Final Incident Report
```

The domain changes.

The pattern remains the same.


# 18. Classroom Exercise - AI Placement Assistant

Now participants should build their own Crew using the same pattern.

## Problem

Create an AI Placement Assistant for final-year engineering students.

Suggested Agents:

1. Resume Reviewer
2. Coding Interview Coach
3. HR Interview Coach
4. Company Research Agent
5. Final Placement Mentor

Final output:

- Resume tips
- Coding preparation plan
- HR interview tips
- Company research checklist
- 7-day preparation plan


In [ ]:
# Exercise Starter Code

# TODO:
# Create your own AI Placement Assistant Crew.

# Suggested steps:
# 1. Create LLM configuration.
# 2. Create 5 Agents.
# 3. Create 5 Tasks.
# 4. Use context chaining.
# 5. Create Crew.
# 6. Run kickoff().


# 19. Challenge Exercise - Add Tools

Extend the Placement Assistant with one custom tool.

Example:

```python
@tool("Interview Resource Tool")
def interview_resource_tool(role: str) -> str:
    ...
```

The tool should return:

- coding practice resources
- resume preparation resources
- HR interview resources
- company research resources


# 20. Capstone Reflection

Think about your own department.

What repetitive academic task could become a Crew of AI specialists?

Examples:

- FDP preparation
- Assignment creation
- Question paper drafting
- Research proposal review
- Placement preparation
- Student mentoring
- Lab manual preparation

Write down:

1. Final output required
2. Specialist Agents needed
3. Task sequence
4. Tools required
5. Quality review step


# 21. Common Mistakes

Avoid these mistakes in capstone-style CrewAI projects:

1. Creating too many Agents without clear roles.
2. Giving vague goals.
3. Forgetting context chaining.
4. Not having a final reviewer.
5. Using tools without testing them.
6. Making the workflow too complex for the first version.
7. Assuming CrewAI replaces human faculty judgment.
8. Forgetting cost and latency during live demos.
9. Not keeping output classroom-friendly.
10. Skipping reflection and discussion.

---

## Instructor Tip

Emphasize this:

> AI prepares a draft. Faculty provides judgment.

That keeps the FDP practical and responsible.


# 22. Key Takeaways

In this capstone, we learned:

- CrewAI helps us think in terms of AI teams.
- Each Agent should have a clear specialist role.
- Tasks should be sequenced logically.
- Context chaining improves coherence.
- Tools give Agents controlled external capability.
- A reviewer Agent improves quality.
- The same architecture can be reused across domains.
- Faculty can adapt this pattern for real academic workflows.

---

## Final Message

Today you did not just learn CrewAI.

You learned how to design an AI team.

That shift is the real foundation of Agentic AI.
